# 07 RQ6 Ablation Study of Framework Components

This notebook evaluates the contribution of the main framework components to
system-level safety and accepted-case performance.

It performs:
- checkpoint loading
- temperature scaling
- Monte Carlo Dropout inference
- Grad-CAM-based explanation support estimation
- comparison of framework variants
- export of tables and figures

Outputs:
- Table 11: ablation comparison of framework variants
- Table 12: component-level contribution summary
- Figure 11: comparative value of framework components
- Figure 12: hybrid CNN–expert system architecture and contribution flow
- ZIP archive of RQ6 outputs

In [1]:
# ----------------------------------------
# Section 1: Imports
# ----------------------------------------

import os
import json
import random
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [2]:
# ----------------------------------------
# Section 2: Reproducibility setup
# ----------------------------------------

SEED = 42

def seed_everything(seed: int = 42) -> None:
    """
    Set random seeds for reproducibility across Python, NumPy, and PyTorch.
    """
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id: int) -> None:
    """
    Ensure each DataLoader worker uses a deterministic seed.
    """
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

seed_everything(SEED)

print("Reproducibility setup completed")
print(f"Global seed: {SEED}")

Reproducibility setup completed
Global seed: 42


In [3]:
# ----------------------------------------
# Section 3: Configuration
# ----------------------------------------

CONFIG = {
    "seed": SEED,
    "image_size": 224,
    "batch_size": 32,
    "num_workers": 0,
    "mc_passes": 30,
    "plantvillage_root": "/kaggle/input/datasets/thedataeng/plantvillage",
    "plantdoc_root": "/kaggle/input/datasets/thedataeng/plantdoc",

    # Update this if your Kaggle dataset input name differs
    "checkpoint_root": "/kaggle/input/datasets/thedataeng/thesis-train-backbones-outputs",

    "model_names": ["resnet50", "efficientnet_b0", "mobilenet_v2"],
    "output_root": "/kaggle/working/thesis_outputs/rq6_ablation",
    "focus_batches": 6,
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_ROOT = Path(CONFIG["output_root"])
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

PRETTY_NAMES = {
    "resnet50": "ResNet50",
    "efficientnet_b0": "EfficientNet-B0",
    "mobilenet_v2": "MobileNetV2",
}

print("Configuration loaded")
print(f"Device: {DEVICE}")
print(f"Models: {CONFIG['model_names']}")
print(f"Output root: {OUTPUT_ROOT}")

Configuration loaded
Device: cuda
Models: ['resnet50', 'efficientnet_b0', 'mobilenet_v2']
Output root: /kaggle/working/thesis_outputs/rq6_ablation


In [4]:
# ----------------------------------------
# Section 4: Helper functions
# ----------------------------------------

def ensure_dir(path: Path) -> Path:
    """
    Create a directory if it does not exist and return the Path object.
    """
    path.mkdir(parents=True, exist_ok=True)
    return path

def get_eval_transform(image_size: int = 224):
    """
    Create the evaluation transform used across RQ6 experiments.
    """
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
    ])

def create_model(model_name: str, num_classes: int):
    """
    Create the selected backbone model and replace the classification head.
    Returns the model and target layer for explanation support estimation.
    """
    if model_name == "resnet50":
        model = models.resnet50(weights=None)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        target_layer = model.layer4[-1].conv3

    elif model_name == "efficientnet_b0":
        model = models.efficientnet_b0(weights=None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
        target_layer = model.features[-1][0]

    elif model_name == "mobilenet_v2":
        model = models.mobilenet_v2(weights=None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
        target_layer = model.features[-1]

    else:
        raise ValueError(f"Unsupported model name: {model_name}")

    return model.to(DEVICE), target_layer

def checkpoint_path(model_name: str) -> Path:
    """
    Return the checkpoint path for the selected model.
    """
    return Path(CONFIG["checkpoint_root"]) / "checkpoints" / f"{model_name}_seed{SEED}_best.pt"

def save_table(df: pd.DataFrame, name: str) -> None:
    """
    Save a DataFrame as CSV in the tables directory.
    """
    table_dir = ensure_dir(OUTPUT_ROOT / "tables")
    csv_path = table_dir / f"{name}.csv"
    df.to_csv(csv_path, index=False)
    print(f"Saved table: {csv_path}")

def save_figure(fig: plt.Figure, name: str) -> None:
    """
    Save a matplotlib figure as PDF in the figures directory.
    """
    fig_dir = ensure_dir(OUTPUT_ROOT / "figures")
    pdf_path = fig_dir / f"{name}.pdf"
    fig.tight_layout()
    fig.savefig(pdf_path, format="pdf", bbox_inches="tight")
    plt.close(fig)
    print(f"Saved figure: {pdf_path}")

def pretty_metric(x: float) -> float:
    """
    Round a metric value for cleaner table presentation.
    """
    return round(float(x), 4)

@torch.no_grad()
def collect_logits_labels(model: nn.Module, loader: DataLoader):
    """
    Collect deterministic logits and labels from a loader.
    """
    model.eval()

    all_logits = []
    all_labels = []

    for images, labels in loader:
        images = images.to(DEVICE)
        logits = model(images)

        all_logits.append(logits.cpu())
        all_labels.append(labels)

    return torch.cat(all_logits, dim=0), torch.cat(all_labels, dim=0)

def fit_temperature(logits: torch.Tensor, labels: torch.Tensor, max_iter: int = 100) -> float:
    """
    Fit a scalar temperature using validation logits and labels.
    """
    temperature = torch.ones(1, requires_grad=True)
    optimizer = torch.optim.LBFGS([temperature], lr=0.01, max_iter=max_iter)
    criterion = nn.CrossEntropyLoss()

    def closure():
        optimizer.zero_grad()
        loss = criterion(logits / temperature.clamp(min=1e-3), labels.long())
        loss.backward()
        return loss

    optimizer.step(closure)
    return float(temperature.detach().item())

def enable_dropout(model: nn.Module) -> None:
    """
    Enable dropout layers during inference for Monte Carlo Dropout.
    """
    for module in model.modules():
        if isinstance(module, nn.Dropout):
            module.train()

@torch.no_grad()
def mc_dropout_predict(model: nn.Module, loader: DataLoader, n_passes: int = 30):
    """
    Perform Monte Carlo Dropout inference across a DataLoader.
    Returns mean probabilities, entropy, predictions, and labels.
    """
    model.eval()
    enable_dropout(model)

    all_mean_probs = []
    all_entropy = []
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images = images.to(DEVICE)

        pass_probs = []
        for _ in range(n_passes):
            logits = model(images)
            probs = torch.softmax(logits, dim=1)
            pass_probs.append(probs.unsqueeze(0))

        pass_probs = torch.cat(pass_probs, dim=0)
        mean_probs = pass_probs.mean(dim=0)
        entropy = -(mean_probs * torch.log(mean_probs.clamp(min=1e-12))).sum(dim=1)
        preds = mean_probs.argmax(dim=1)

        all_mean_probs.append(mean_probs.cpu().numpy())
        all_entropy.append(entropy.cpu().numpy())
        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.numpy())

    return {
        "mean_probs": np.vstack(all_mean_probs),
        "entropy": np.concatenate(all_entropy),
        "preds": np.concatenate(all_preds),
        "labels": np.concatenate(all_labels),
    }

def normalize_entropy(entropy: np.ndarray, num_classes: int) -> np.ndarray:
    """
    Normalize entropy to the range [0, 1].
    """
    max_entropy = np.log(num_classes + 1e-12)
    return entropy / max_entropy

def denormalize(image_tensor: torch.Tensor) -> torch.Tensor:
    """
    Convert a normalized image tensor back to the [0, 1] range.
    """
    mean = torch.tensor([0.485, 0.456, 0.406], device=image_tensor.device).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=image_tensor.device).view(3, 1, 1)
    return torch.clamp(image_tensor * std + mean, 0, 1)

def simple_leaf_mask(image_np: np.ndarray) -> np.ndarray:
    """
    Create a heuristic foreground mask approximating the leaf region using RGB conditions.
    """
    r = image_np[:, :, 0]
    g = image_np[:, :, 1]
    b = image_np[:, :, 2]
    mask = (g > r * 0.85) & (g > b * 0.85) & (g > 0.2)
    return mask.astype(np.uint8)

def safe_score(correct: np.ndarray, accepted: np.ndarray) -> float:
    """
    Compute decision safety as 1 - (unsafe accepted predictions / total samples).
    """
    unsafe_accepts = int(((~correct) & accepted).sum())
    total = len(correct)
    return 1.0 - unsafe_accepts / total

def explanation_focus_proxy(model: nn.Module, target_layer: nn.Module, loader: DataLoader, limit_batches: int = 6):
    """
    Estimate explanation support using a lightweight Grad-CAM concentration analysis.
    Returns one focus value per evaluated sample.
    """
    activations = []
    gradients = []

    def fwd_hook(module, inputs, outputs):
        activations.append(outputs.detach())

    def bwd_hook(module, grad_input, grad_output):
        gradients.append(grad_output[0].detach())

    h1 = target_layer.register_forward_hook(fwd_hook)
    h2 = target_layer.register_full_backward_hook(bwd_hook)

    focus_values = []
    batches_seen = 0

    for images, labels in loader:
        images = images.to(DEVICE)

        model.zero_grad(set_to_none=True)
        logits = model(images)
        class_idx = logits.argmax(dim=1)

        score = logits[torch.arange(logits.size(0)), class_idx].sum()
        score.backward()

        acts = activations[-1]
        grads = gradients[-1]

        weights = grads.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * acts).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=images.shape[-2:], mode="bilinear", align_corners=False).squeeze(1)

        cam = cam - cam.amin(dim=(1, 2), keepdim=True)
        cam = cam / (cam.amax(dim=(1, 2), keepdim=True) + 1e-8)

        for i in range(images.size(0)):
            image_np = denormalize(images[i]).detach().cpu().permute(1, 2, 0).numpy()
            leaf_mask = simple_leaf_mask(image_np)

            cam_np = cam[i].detach().cpu().numpy()
            focus = float((cam_np * leaf_mask).sum() / (cam_np.sum() + 1e-8))
            focus_values.append(focus)

        batches_seen += 1
        print(f"  Processed focus batch {batches_seen}/{limit_batches}")

        if batches_seen >= limit_batches:
            break

    h1.remove()
    h2.remove()

    return np.array(focus_values)

def expert_decision_with_explanation(conf: float, ent: float, focus: float) -> tuple:
    """
    Apply the final expert decision-support logic using confidence, uncertainty,
    and explanation support.
    """
    # Strongest acceptance condition
    if conf >= 0.75 and ent <= 0.30 and focus >= 0.80:
        return "Accept", "Recommend treatment or monitoring by predicted class"

    # Moderately reliable case
    if conf >= 0.60 and ent <= 0.40 and focus >= 0.60:
        return "Monitor", "Monitor and retake if symptoms persist"

    # Some usable evidence exists, but reliability is insufficient
    if conf >= 0.50 and ent <= 0.50 and focus >= 0.50:
        return "Retake", "Request a clearer image"

    # Lowest-confidence / highest-risk case
    return "Review", "Seek expert review"
    
print('Done')

Done


In [5]:
# ----------------------------------------
# Section 5: Dataset loading
# ----------------------------------------

pv_root = Path(CONFIG["plantvillage_root"])
pd_root = Path(CONFIG["plantdoc_root"])

val_dir = pv_root / "val"

assert val_dir.exists(), f"Missing PlantVillage validation directory: {val_dir}"
assert pd_root.exists(), f"Missing PlantDoc directory: {pd_root}"

eval_tfms = get_eval_transform(CONFIG["image_size"])

val_dataset = datasets.ImageFolder(val_dir, transform=eval_tfms)
plantdoc_dataset = datasets.ImageFolder(pd_root, transform=eval_tfms)

generator = torch.Generator()
generator.manual_seed(SEED)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    worker_init_fn=seed_worker,
    generator=generator,
    pin_memory=True,
)

plantdoc_loader = DataLoader(
    plantdoc_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    worker_init_fn=seed_worker,
    generator=generator,
    pin_memory=True,
)

CLASS_NAMES = plantdoc_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)

print("Datasets loaded successfully")
print(f"Validation samples: {len(val_dataset)}")
print(f"PlantDoc samples:   {len(plantdoc_dataset)}")
print(f"Number of classes:  {NUM_CLASSES}")

Datasets loaded successfully
Validation samples: 5518
PlantDoc samples:   2555
Number of classes:  27


In [6]:
# ----------------------------------------
# Section 6: Checkpoint loading
# ----------------------------------------

models_loaded = {}
target_layers = {}

for idx, model_name in enumerate(CONFIG["model_names"], start=1):
    print(f"Loading checkpoint {idx}/{len(CONFIG['model_names'])}: {model_name}")

    ckpt_path = checkpoint_path(model_name)
    assert ckpt_path.exists(), f"Missing checkpoint: {ckpt_path}"

    model, target_layer = create_model(model_name, NUM_CLASSES)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval()

    models_loaded[model_name] = model
    target_layers[model_name] = target_layer

    print(f"Loaded checkpoint: {ckpt_path}")

print("All checkpoints loaded successfully")

Loading checkpoint 1/3: resnet50
Loaded checkpoint: /kaggle/input/datasets/thedataeng/thesis-train-backbones-outputs/checkpoints/resnet50_seed42_best.pt
Loading checkpoint 2/3: efficientnet_b0
Loaded checkpoint: /kaggle/input/datasets/thedataeng/thesis-train-backbones-outputs/checkpoints/efficientnet_b0_seed42_best.pt
Loading checkpoint 3/3: mobilenet_v2
Loaded checkpoint: /kaggle/input/datasets/thedataeng/thesis-train-backbones-outputs/checkpoints/mobilenet_v2_seed42_best.pt
All checkpoints loaded successfully


In [7]:
# ----------------------------------------
# Section 7: Calibration and Monte Carlo Dropout preparation for all models
# ----------------------------------------

temperature_map = {}
deterministic_results = {}
mc_results_map = {}

for idx, model_name in enumerate(CONFIG["model_names"], start=1):
    print(f"Preparing calibration and MC Dropout results for model {idx}/{len(CONFIG['model_names'])}: {model_name}")

    model = models_loaded[model_name]

    print("  Collecting validation logits")
    val_logits, val_labels = collect_logits_labels(model, val_loader)

    print("  Fitting temperature parameter")
    temperature = fit_temperature(val_logits, val_labels)
    temperature_map[model_name] = temperature

    print("  Collecting PlantDoc logits")
    pd_logits, pd_labels_t = collect_logits_labels(model, plantdoc_loader)
    pd_labels = pd_labels_t.numpy()

    probs_calibrated = torch.softmax(pd_logits / temperature, dim=1).numpy()
    preds_calibrated = probs_calibrated.argmax(axis=1)
    conf_calibrated = probs_calibrated.max(axis=1)

    deterministic_results[model_name] = {
        "pd_labels": pd_labels,
        "probs_calibrated": probs_calibrated,
        "preds_calibrated": preds_calibrated,
        "conf_calibrated": conf_calibrated,
    }

    print("  Running Monte Carlo Dropout inference")
    mc_results = mc_dropout_predict(
        model=model,
        loader=plantdoc_loader,
        n_passes=CONFIG["mc_passes"]
    )

    mc_results_map[model_name] = {
        "mean_probs": mc_results["mean_probs"],
        "entropy": mc_results["entropy"],
        "norm_entropy": normalize_entropy(mc_results["entropy"], NUM_CLASSES),
        "preds": mc_results["preds"],
        "labels": mc_results["labels"],
    }

    print(f"  Temperature: {temperature:.4f}")

print("Calibration and MC Dropout preparation completed successfully for all models")

Preparing calibration and MC Dropout results for model 1/3: resnet50
  Fitting temperature parameter
  Running Monte Carlo Dropout inference
  Temperature: 1.0020
Preparing calibration and MC Dropout results for model 2/3: efficientnet_b0
  Fitting temperature parameter
  Running Monte Carlo Dropout inference
  Temperature: 1.1691
Preparing calibration and MC Dropout results for model 3/3: mobilenet_v2
  Fitting temperature parameter
  Running Monte Carlo Dropout inference
  Temperature: 0.9974
Calibration and MC Dropout preparation completed successfully for all models


In [8]:
# ----------------------------------------
# Section 8: Explanation support estimation for all models
# ----------------------------------------

focus_map = {}

for idx, model_name in enumerate(CONFIG["model_names"], start=1):
    print(f"Estimating explanation support for model {idx}/{len(CONFIG['model_names'])}: {model_name}")

    model = models_loaded[model_name]
    target_layer = target_layers[model_name]

    focus_values = explanation_focus_proxy(
        model=model,
        target_layer=target_layer,
        loader=plantdoc_loader,
        limit_batches=CONFIG["focus_batches"]
    )

    if len(focus_values) < len(deterministic_results[model_name]["preds_calibrated"]):
        mean_focus = np.nanmean(focus_values) if len(focus_values) > 0 else 0.0
        focus_values = np.pad(
            focus_values,
            (0, len(deterministic_results[model_name]["preds_calibrated"]) - len(focus_values)),
            constant_values=np.nan
        )
        focus_values = np.where(np.isnan(focus_values), mean_focus, focus_values)

    focus_map[model_name] = focus_values

print("Explanation support estimation completed successfully for all models")

Estimating explanation support for model 1/3: resnet50
  Processed focus batch 1/6
  Processed focus batch 2/6
  Processed focus batch 3/6
  Processed focus batch 4/6
  Processed focus batch 5/6
  Processed focus batch 6/6
Estimating explanation support for model 2/3: efficientnet_b0
  Processed focus batch 1/6
  Processed focus batch 2/6
  Processed focus batch 3/6
  Processed focus batch 4/6
  Processed focus batch 5/6
  Processed focus batch 6/6
Estimating explanation support for model 3/3: mobilenet_v2
  Processed focus batch 1/6
  Processed focus batch 2/6
  Processed focus batch 3/6
  Processed focus batch 4/6
  Processed focus batch 5/6
  Processed focus batch 6/6
Explanation support estimation completed successfully for all models


In [9]:
# ----------------------------------------
# Section 9: Define ablation variants for all models
# ----------------------------------------

variant_rows = []

for idx, model_name in enumerate(CONFIG["model_names"], start=1):
    print(f"Building ablation variants for model {idx}/{len(CONFIG['model_names'])}: {model_name}")

    det = deterministic_results[model_name]
    mc = mc_results_map[model_name]
    focus_values = focus_map[model_name]

    pd_labels = det["pd_labels"]
    preds_calibrated = det["preds_calibrated"]
    conf_calibrated = det["conf_calibrated"]
    norm_entropy = mc["norm_entropy"]

    correct = preds_calibrated == pd_labels

    # Shared explanation-support definition
    exp_mask = focus_values >= 0.75

    # -------------------------------
    # Variant 1: CNN only
    # -------------------------------
    accepted_cnn = np.ones_like(correct, dtype=bool)
    deferred_cnn = ~accepted_cnn

    variant_rows.append({
        "Model": model_name,
        "System Variant": "CNN only",
        "Accuracy": pretty_metric(correct.mean()),
        "Accepted Accuracy": pretty_metric((correct & accepted_cnn).sum() / max(1, accepted_cnn.sum())),
        "Decision Safety": pretty_metric(safe_score(correct, accepted_cnn)),
        "Deferral Rate": pretty_metric(deferred_cnn.mean()),
        "Unsafe Accepts": int(((~correct) & accepted_cnn).sum()),
        "Correct Escalations": int((deferred_cnn & (~correct)).sum()),
        "Explanation Support Rate": 0.0
    })

    # -------------------------------
    # Variant 2: CNN + calibrated confidence
    # -------------------------------
    accepted_conf = conf_calibrated >= 0.60
    deferred_conf = ~accepted_conf

    variant_rows.append({
        "Model": model_name,
        "System Variant": "CNN + calibrated confidence",
        "Accuracy": pretty_metric(correct.mean()),
        "Accepted Accuracy": pretty_metric((correct & accepted_conf).sum() / max(1, accepted_conf.sum())),
        "Decision Safety": pretty_metric(safe_score(correct, accepted_conf)),
        "Deferral Rate": pretty_metric(deferred_conf.mean()),
        "Unsafe Accepts": int(((~correct) & accepted_conf).sum()),
        "Correct Escalations": int((deferred_conf & (~correct)).sum()),
        "Explanation Support Rate": 0.0
    })

    # -------------------------------
    # Variant 3: CNN + confidence + uncertainty
    # -------------------------------
    accepted_unc = (conf_calibrated >= 0.60) & (norm_entropy <= 0.40)
    deferred_unc = ~accepted_unc

    variant_rows.append({
        "Model": model_name,
        "System Variant": "CNN + confidence + MC Dropout uncertainty",
        "Accuracy": pretty_metric(correct.mean()),
        "Accepted Accuracy": pretty_metric((correct & accepted_unc).sum() / max(1, accepted_unc.sum())),
        "Decision Safety": pretty_metric(safe_score(correct, accepted_unc)),
        "Deferral Rate": pretty_metric(deferred_unc.mean()),
        "Unsafe Accepts": int(((~correct) & accepted_unc).sum()),
        "Correct Escalations": int((deferred_unc & (~correct)).sum()),
        "Explanation Support Rate": 0.0
    })

    # -------------------------------
    # Variant 4: CNN + confidence + uncertainty + explanation
    # -------------------------------
    accepted_exp = (conf_calibrated >= 0.60) & (norm_entropy <= 0.40) & exp_mask
    deferred_exp = ~accepted_exp

    variant_rows.append({
        "Model": model_name,
        "System Variant": "CNN + confidence + uncertainty + explanation support",
        "Accuracy": pretty_metric(correct.mean()),
        "Accepted Accuracy": pretty_metric((correct & accepted_exp).sum() / max(1, accepted_exp.sum())),
        "Decision Safety": pretty_metric(safe_score(correct, accepted_exp)),
        "Deferral Rate": pretty_metric(deferred_exp.mean()),
        "Unsafe Accepts": int(((~correct) & accepted_exp).sum()),
        "Correct Escalations": int((deferred_exp & (~correct)).sum()),
        "Explanation Support Rate": pretty_metric((exp_mask & accepted_exp).sum() / max(1, accepted_exp.sum())),
    })

    # -------------------------------
    # Variant 5: Full expert decision layer
    # -------------------------------
    actions = []
    for conf, ent, focus in zip(conf_calibrated, norm_entropy, focus_values):
        action, _ = expert_decision_with_explanation(float(conf), float(ent), float(focus))
        actions.append(action)

    actions = np.array(actions)

    accepted_full = actions == "Accept"
    deferred_full = ~accepted_full

    variant_rows.append({
        "Model": model_name,
        "System Variant": "CNN + confidence + uncertainty + explanation + expert decision layer",
        "Accuracy": pretty_metric(correct.mean()),
        "Accepted Accuracy": pretty_metric((correct & accepted_full).sum() / max(1, accepted_full.sum())),
        "Decision Safety": pretty_metric(safe_score(correct, accepted_full)),
        "Deferral Rate": pretty_metric(deferred_full.mean()),
        "Unsafe Accepts": int(((~correct) & accepted_full).sum()),
        "Correct Escalations": int((deferred_full & (~correct)).sum()),
        "Explanation Support Rate": pretty_metric((exp_mask & accepted_full).sum() / max(1, accepted_full.sum())),
    })

ablation_df = pd.DataFrame(variant_rows)
ablation_df["Model"] = ablation_df["Model"].map(PRETTY_NAMES)

print("Ablation variants computed successfully for all models")
display(ablation_df.head(15))

Building ablation variants for model 1/3: resnet50
Building ablation variants for model 2/3: efficientnet_b0
Building ablation variants for model 3/3: mobilenet_v2
Ablation variants computed successfully for all models


,Model,System Variant,Accuracy,Accepted Accuracy,Decision Safety,Deferral Rate,Unsafe Accepts,Correct Escalations,Explanation Support Rate
0,ResNet50,CNN only,0.2023,0.2023,0.2023,0.0000,2038,0,0.0
1,ResNet50,CNN + calibrated confidence,0.2023,0.2339,0.4654,0.3022,1366,672,0.0
2,ResNet50,CNN + confidence + MC Dropout uncertainty,0.2023,0.2344,0.4771,0.3170,1336,702,0.0
3,ResNet50,CNN + confidence + uncertainty + explanation s...,0.2023,0.2388,0.4896,0.3295,1304,734,1.0
4,ResNet50,CNN + confidence + uncertainty + explanation +...,0.2023,0.2566,0.6031,0.4661,1014,1024,1.0
5,EfficientNet-B0,CNN only,0.1765,0.1765,0.1765,0.0000,2104,0,0.0
6,EfficientNet-B0,CNN + calibrated confidence,0.1765,0.2263,0.5303,0.3930,1200,904,0.0
7,EfficientNet-B0,CNN + confidence + MC Dropout uncertainty,0.1765,0.2263,0.5303,0.3930,1200,904,0.0
8,EfficientNet-B0,CNN + confidence + uncertainty + explanation s...,0.1765,0.2274,0.5346,0.3977,1189,915,1.0
9,EfficientNet-B0,CNN + confidence + uncertainty + explanation +...,0.1765,0.2620,0.6802,0.5667,817,1287,1.0


In [10]:
# ----------------------------------------
# Section 10: Save full ablation results table
# ----------------------------------------

save_table(ablation_df, "Full_Ablation_Results_All_Columns")

print("Full ablation results table saved successfully")


Saved table: /kaggle/working/thesis_outputs/rq6_ablation/tables/Full_Ablation_Results_All_Columns.csv
Full ablation results table saved successfully


In [11]:
# ----------------------------------------
# Section 11: Save Table 11 - Ablation comparison
# ----------------------------------------

table11_df = ablation_df[
    ["Model", "System Variant", "Accuracy", "Accepted Accuracy", "Decision Safety", "Deferral Rate"]
].copy()

save_table(table11_df, "Table_11_Ablation_Framework_Variants")

print("Table 11 saved successfully")
display(table11_df)

Saved table: /kaggle/working/thesis_outputs/rq6_ablation/tables/Table_11_Ablation_Framework_Variants.csv
Table 11 saved successfully


,Model,System Variant,Accuracy,Accepted Accuracy,Decision Safety,Deferral Rate
0,ResNet50,CNN only,0.2023,0.2023,0.2023,0.0000
1,ResNet50,CNN + calibrated confidence,0.2023,0.2339,0.4654,0.3022
2,ResNet50,CNN + confidence + MC Dropout uncertainty,0.2023,0.2344,0.4771,0.3170
3,ResNet50,CNN + confidence + uncertainty + explanation s...,0.2023,0.2388,0.4896,0.3295
4,ResNet50,CNN + confidence + uncertainty + explanation +...,0.2023,0.2566,0.6031,0.4661
5,EfficientNet-B0,CNN only,0.1765,0.1765,0.1765,0.0000
6,EfficientNet-B0,CNN + calibrated confidence,0.1765,0.2263,0.5303,0.3930
7,EfficientNet-B0,CNN + confidence + MC Dropout uncertainty,0.1765,0.2263,0.5303,0.3930
8,EfficientNet-B0,CNN + confidence + uncertainty + explanation s...,0.1765,0.2274,0.5346,0.3977
9,EfficientNet-B0,CNN + confidence + uncertainty + explanation +...,0.1765,0.2620,0.6802,0.5667


In [15]:
# ----------------------------------------
# Section 11: Save Table 12 - Component-level contribution
# ----------------------------------------

table12_df = ablation_df[
    ["Model", "System Variant", "Unsafe Accepts", "Correct Escalations", "Explanation Support Rate"]
].copy()

def interpret_variant(name: str) -> str:
    """
    Provide a short interpretation for each ablation variant.
    """
    if name == "CNN only":
        return "High risk, no decision filtering"
    elif name == "CNN + calibrated confidence":
        return "Moderate filtering improvement"
    elif name == "CNN + confidence + MC Dropout uncertainty":
        return "Stronger uncertainty-aware filtering"
    elif name == "CNN + confidence + uncertainty + explanation support":
        return "Better interpretability of risky cases"
    return "Best overall safety and decision support"

table12_df["Overall Interpretation"] = table12_df["System Variant"].apply(interpret_variant)

save_table(table12_df, "Table_12_Component_Level_Contribution")

print("Table 12 saved successfully")
display(table12_df)

Saved table: /kaggle/working/thesis_outputs/rq6_ablation/tables/Table_12_Component_Level_Contribution.csv
Table 12 saved successfully


,Model,System Variant,Unsafe Accepts,Correct Escalations,Explanation Support Rate,Overall Interpretation
0,ResNet50,CNN only,2038,0,0.0,"High risk, no decision filtering"
1,ResNet50,CNN + calibrated confidence,1366,672,0.0,Moderate filtering improvement
2,ResNet50,CNN + confidence + MC Dropout uncertainty,1336,702,0.0,Stronger uncertainty-aware filtering
3,ResNet50,CNN + confidence + uncertainty + explanation s...,1304,734,1.0,Better interpretability of risky cases
4,ResNet50,CNN + confidence + uncertainty + explanation +...,1014,1024,1.0,Best overall safety and decision support
5,EfficientNet-B0,CNN only,2104,0,0.0,"High risk, no decision filtering"
6,EfficientNet-B0,CNN + calibrated confidence,1200,904,0.0,Moderate filtering improvement
7,EfficientNet-B0,CNN + confidence + MC Dropout uncertainty,1200,904,0.0,Stronger uncertainty-aware filtering
8,EfficientNet-B0,CNN + confidence + uncertainty + explanation s...,1189,915,1.0,Better interpretability of risky cases
9,EfficientNet-B0,CNN + confidence + uncertainty + explanation +...,817,1287,1.0,Best overall safety and decision support


In [16]:
# ----------------------------------------
# Section 12: Figure 11 - Framework component comparison
# ----------------------------------------

fig, ax = plt.subplots(figsize=(14, 6))

variant_order = [
    "CNN only",
    "CNN + calibrated confidence",
    "CNN + confidence + MC Dropout uncertainty",
    "CNN + confidence + uncertainty + explanation support",
    "CNN + confidence + uncertainty + explanation + expert decision layer",
]

model_order = ["ResNet50", "EfficientNet-B0", "MobileNetV2"]
metric_order = ["Accuracy", "Accepted Accuracy", "Decision Safety"]

# One color per model
model_colors = {
    "ResNet50": "C0",
    "EfficientNet-B0": "C1",
    "MobileNetV2": "C2",
}

# for each variant: 3 models x 3 metrics
n_models = len(model_order)
n_metrics = len(metric_order)
bars_per_group = n_models * n_metrics
group_centers = np.arange(len(variant_order)) * (bars_per_group + 2)
bar_width = 0.8

from matplotlib.patches import Patch

# Short text labels inside bars
metric_text = {
    "Accuracy": "Acc",
    "Accepted Accuracy": "Accepted Acc",
    "Decision Safety": "Decition Safe",
}

for g, variant in enumerate(variant_order):
    base = group_centers[g]
    subset = ablation_df[ablation_df["System Variant"] == variant]

    pos = 0
    for model_name in model_order:
        row = subset[subset["Model"] == model_name].iloc[0]

        for metric in metric_order:
            x = base + pos
            y = row[metric]

            ax.bar(
                x,
                y,
                width=bar_width,
                color=model_colors[model_name],
                edgecolor="black",
                linewidth=0.6,
            )

            # metric name inside bar
            ax.text(
                x,
                max(y * 0.55, 0.03),
                metric_text[metric],
                ha="center",
                va="center",
                fontsize=7,
                rotation=90,
                color="white",
                fontweight="bold"
            )

            # value above bar
            ax.text(
                x,
                y + 0.012,
                f"{y:.2f}",
                ha="center",
                va="bottom",
                fontsize=7,
                rotation=90
            )

            pos += 1

# Legend only for models
model_handles = [
    Patch(facecolor=model_colors[m], edgecolor="black", label=m)
    for m in model_order
]
ax.legend(handles=model_handles, title="Model", loc="upper left", frameon=True)

# One tick per variant group
tick_positions = [c + (bars_per_group - 1) / 2 for c in group_centers]
short_labels = [
    "CNN only",
    "+ Confidence",
    "+ Uncertainty",
    "+ Explanation",
    "+ Expert Layer",
]

ax.set_xticks(tick_positions)
ax.set_xticklabels(short_labels, rotation=15, ha="right")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.0)
ax.set_title("Figure 11. Comparative Value of Framework Components in System-Level Decision Support")
ax.grid(axis="y", alpha=0.25)

save_figure(fig, "Figure_11_Framework_Component_Comparison")

Saved figure: /kaggle/working/thesis_outputs/rq6_ablation/figures/Figure_11_Framework_Component_Comparison.pdf


In [17]:
# ----------------------------------------
# Section 13: Figure 12 - Behavioral contribution across framework variants
# ----------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), sharey=True)

model_order = ["ResNet50", "EfficientNet-B0", "MobileNetV2"]
variant_order = [
    "CNN only",
    "CNN + calibrated confidence",
    "CNN + confidence + MC Dropout uncertainty",
    "CNN + confidence + uncertainty + explanation support",
    "CNN + confidence + uncertainty + explanation + expert decision layer",
]

short_labels = [
    "CNN only",
    "+ Confidence",
    "+ Uncertainty",
    "+ Explanation",
    "+ Expert Layer",
]

for ax, model_name in zip(axes, model_order):
    model_df = table12_df[table12_df["Model"] == model_name].copy()
    model_df = model_df.set_index("System Variant").loc[variant_order].reset_index()

    x = np.arange(len(model_df))

    ax.plot(
        x,
        model_df["Unsafe Accepts"],
        marker="o",
        linewidth=2,
        label="Unsafe Accepts"
    )

    ax.plot(
        x,
        model_df["Correct Escalations"],
        marker="s",
        linewidth=2,
        label="Correct Escalations"
    )

    ax.set_title(model_name)
    ax.set_xticks(x)
    ax.set_xticklabels(short_labels, rotation=18, ha="right")
    ax.grid(alpha=0.25)

axes[0].set_ylabel("Count")
axes[0].legend(frameon=True)

fig.suptitle("Figure 12. Behavioral Contribution Across Framework Variants", y=1.03)

save_figure(fig, "Figure_12_Component_Level_Behavioral_Contribution")

Saved figure: /kaggle/working/thesis_outputs/rq6_ablation/figures/Figure_12_Component_Level_Behavioral_Contribution.pdf


In [18]:
# ----------------------------------------
# Section 14: Save RQ6 metadata
# ----------------------------------------

meta_dir = ensure_dir(OUTPUT_ROOT / "metadata")

rq6_meta = {
    "seed": SEED,
    "models_evaluated": CONFIG["model_names"],
    "temperatures": {k: pretty_metric(v) for k, v in temperature_map.items()},
    "mc_passes": CONFIG["mc_passes"],
    "focus_batches": CONFIG["focus_batches"],
    "num_classes": NUM_CLASSES,
    "num_samples": len(deterministic_results[CONFIG["model_names"][0]]["pd_labels"]),
}

meta_path = meta_dir / "rq6_metadata.json"
with open(meta_path, "w") as f:
    json.dump(rq6_meta, f, indent=2)

print("RQ6 metadata saved successfully")
print(f"Metadata path: {meta_path}")

RQ6 metadata saved successfully
Metadata path: /kaggle/working/thesis_outputs/rq6_ablation/metadata/rq6_metadata.json


In [19]:
# ----------------------------------------
# Section 15: Create ZIP archive
# ----------------------------------------

zip_path = OUTPUT_ROOT.parent / "07_rq6_ablation_outputs.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in OUTPUT_ROOT.rglob("*"):
        if file_path.is_file():
            zf.write(file_path, arcname=file_path.relative_to(OUTPUT_ROOT))

print("ZIP archive created successfully")
print(f"ZIP file: {zip_path}")
print("07_rq6_ablation notebook completed successfully")

ZIP archive created successfully
ZIP file: /kaggle/working/thesis_outputs/07_rq6_ablation_outputs.zip
07_rq6_ablation notebook completed successfully
